# FFT MLOps Demo: Step 1 - Simple Model Training
This notebook demonstrates the initial stage of our MLOps pipeline: generating synthetic signal data, performing Fourier transformation (FFT) to extract features, and training a simple classification model.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.fft as fft
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
import wandb

# Set seed for reproducibility
np.random.seed(42)

## 1. Data Generation
We generate synthetic time-series signals. Each signal contains two sine waves of different frequencies. Our goal is to classify the signal based on its frequency components using FFT features.

In [ ]:
def generate_signals(n_samples=200, duration=1.0, fs=1000):
    t = np.linspace(0, duration, int(fs * duration), endpoint=False)
    X = []
    y = []
    
    for _ in range(n_samples):
        # Class 0: Low frequencies (e.g., 50Hz and 120Hz)
        # Class 1: High frequencies (e.g., 200Hz and 300Hz)
        is_class_1 = np.random.choice([0, 1])
        
        if is_class_1 == 0:
            f1, f2 = 50, 120
        else:
            f1, f2 = 200, 300
            
        # Generate signal: sin(f1) + sin(f2) + noise
        noise = np.random.normal(0, 0.5, len(t))
        signal = np.sin(2 * np.pi * f1 * t) + np.sin(2 * np.pi * f2 * t) + noise
        
        # Compute FFT
        yf = fft.fft(signal)
        xf = fft.fftfreq(len(t), 1/fs)
        
        # We only take the positive frequencies and their magnitudes
        pos_mask = xf > 0
        magnitudes = np.abs(yf[pos_mask])
        
        X.append(magnitudes)
        y.append(is_class_1)
    
    return np.array(X), np.array(y), xf[pos_mask], t

fs = 1000 # Sampling frequency
X, y, frequencies, time = generate_signals(n_samples=300, fs=fs)
print(f"Generated {X.shape[0]} samples with {X.shape[1]} FFT features.")

## 2. Model Training with Experiment Tracking
We use `wandb` to track our experiment parameters and metrics like accuracy.

In [ ]:
# Initialize wandb
wandb.init(project="fft-mlops-demo", name="simple-rf-baseline")

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define model
model = RandomForestClassifier(n_estimators=100, random_state=42)

# Train
model.fit(X_train, y_train)

# Predict and evaluate
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

# Log to wandb
wandb.log({"accuracy": acc})
print(f"Model Accuracy: {acc:.4f}")

## 3. Visualization
Visualizing the signal and the prediction results.

In [ ]:
# Plot a sample signal from the test set
idx = 0
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(time[:200], X[idx][:200]) # Just showing a snippet of the FFT magnitude
plt.title("FFT Magnitude (Subset)")
plt.xlabel("Frequency Index")
plt.ylabel("Magnitude")

plt.subplot(1, 2, 2)
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix")
plt.show()

wandb.finish()